## 0) Configuration (edit these paths first)

# UFM-Transformer Kaggle Orchestration Notebook

This notebook orchestrates the project execution based on `instructions_integration.txt`.

✅ **Step 2 is skipped** (dataset download), because the dataset is assumed to already exist in the Kaggle environment.


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil
import os
import itertools

# =========================
# USER CONFIGURATION
# =========================
# Path where this repository is available in Kaggle (Working or Input).
# Example: Path('/kaggle/working/Ngbayafou_Deep Learning Biométrie Multimodale')
PROJECT_ROOT = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate')

# ---------------------------------------------------------------------------
# Dataset configuration — choose ONE of the two modes below:
# ---------------------------------------------------------------------------

# MODE A: Combined dataset (face + fingerprint share the same subject structure)
#   Example: Path('/kaggle/input/socofing')
DATASET_PATH = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate/data')

# MODE B: Separate datasets (CASIA-WebFace + SOCOFing, no shared subjects)
#   Set both paths to enable separate-dataset mode.
#   Leave as None to use MODE A.
FACE_PATH = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate/data/casia-webface-extracted')
FINGERPRINT_PATH = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate/data/SOCOFing/Real')

# Outputs in Kaggle working directory
OUTPUT_DIR = Path('/kaggle/working/output')
RESULTS_DIR = Path('/kaggle/working/results')

# Training/eval parameters
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
EMBED_DIM = 512
NUM_WORKERS = 4
DEVICE = 'auto'  # auto, cuda, cpu

# Resume configuration for training step
# Set RESUME_FROM manually to a checkpoint path, or keep None and AUTO_RESUME=True.
RESUME_FROM = Path('/kaggle/working/output/best_model_phase2.pt')
AUTO_RESUME = False

# -------------------------------------------------------------
# Timeout: gracefully stop training before Kaggle's 30h limit.
# The training process will receive a SIGTERM, finish the
# current epoch, save a checkpoint, and exit cleanly.
# Set to None to disable (run until natural completion).
TIMEOUT_SECONDS = 29 * 3600  # 29 hours = 104400 seconds

SRC_DIR = Path('/kaggle/input/datasets/lynettecheripha/ufm-code-ddp-3')
LATEX_DIR = Path('/kaggle/working/latex')

def run(cmd, cwd=None, check=True):
    print('\n' + '='*100)
    print('RUN:', ' '.join(str(x) for x in cmd))
    print('CWD:', cwd if cwd else os.getcwd())
    print('='*100)
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with return code {result.returncode}: {cmd}')
    return result.returncode

USE_SEPARATE = FACE_PATH is not None and FINGERPRINT_PATH is not None

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_DIR:', SRC_DIR)
print('DATASET_PATH:', DATASET_PATH)
print('FACE_PATH:', FACE_PATH)
print('FINGERPRINT_PATH:', FINGERPRINT_PATH)
print('USE_SEPARATE:', USE_SEPARATE)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('TEMPLATE_LATEX_DIR:', TEMPLATE_LATEX_DIR)

# Copy LaTeX template directory to working directory so it is writable
TEMPLATE_LATEX_DIR = PROJECT_ROOT / 'latex'  # Adjust path if template is uploaded in a separate dataset
source_latex = TEMPLATE_LATEX_DIR
if source_latex.exists():
    print(f'Copying latex directory from {source_latex} to {LATEX_DIR}...')
    if LATEX_DIR.exists():
        shutil.rmtree(LATEX_DIR)
    shutil.copytree(source_latex, LATEX_DIR)
    print('Successfully copied latex directory.')
else:
    print(f'Source latex directory {source_latex} not found.')

In [ ]:
# Safety switch: set to True only when you really want to clear OUTPUT_DIR.
# CLEAR_OUTPUT_DIR = False

# if CLEAR_OUTPUT_DIR:
#     if OUTPUT_DIR.exists():
#         print('Clearing OUTPUT_DIR:', OUTPUT_DIR)
#         for item in OUTPUT_DIR.iterdir():
#             if item.is_dir():
#                 shutil.rmtree(item)
#             else:
#                 item.unlink()
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     print('OUTPUT_DIR cleared.')
# else:
#     print('CLEAR_OUTPUT_DIR is False; keeping existing OUTPUT_DIR contents.')
#     if OUTPUT_DIR.exists():
#         existing = sorted(OUTPUT_DIR.iterdir())
#         print(f'Existing items: {len(existing)}')
#         for item in existing[:20]:
#             print(' -', item)
#     else:
#         print('OUTPUT_DIR does not exist yet:', OUTPUT_DIR)


In [ ]:
# Move/Copy pre-trained checkpoints from input to working directory
import shutil
from pathlib import Path

source_dir = Path('/kaggle/input/datasets/lynettecheripha/ufm-outputs')
target_dir = Path('/kaggle/working/output')

if source_dir.exists():
    print(f'Copying checkpoints from {source_dir} to {target_dir}...')
    target_dir.mkdir(parents=True, exist_ok=True)
    copied_count = 0
    for item in source_dir.glob('**/*'):
        if item.is_file():
            rel_path = item.relative_to(source_dir)
            dest_file = target_dir / rel_path
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest_file)
            copied_count += 1
            print(f'  Copied: {rel_path}')
    print(f'Successfully copied {copied_count} files.')
else:
    print(f'Source checkpoints directory {source_dir} not found. Skipping.')


## 1) Analyze and verify project structure

In [ ]:
required_paths = [
    PROJECT_ROOT,
    SRC_DIR,
    SRC_DIR / 'main.py',
    SRC_DIR / 'requirements.txt',
    LATEX_DIR,
    TEMPLATE_LATEX_DIR,
]

for p in required_paths:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

print('\nTop-level content:')
for p in sorted(PROJECT_ROOT.iterdir()):
    print(' -', p.name)

print('\nSource files in src/:')
for p in sorted(SRC_DIR.glob('*.py')):
    print(' -', p.name)


## 2) Environment setup (Step 1 from the guide)

In [ ]:
# Install project dependencies (Kaggle-safe)
# We intentionally avoid upgrading pip/system packages because Kaggle
# preinstalls RAPIDS components that may report resolver conflicts.
rc = run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade-strategy', 'only-if-needed',
    '-r', str(SRC_DIR / 'requirements.txt')
], check=False)
if rc != 0:
    raise RuntimeError('Dependency installation failed. See logs above.')

print('Note: pip dependency-conflict warnings with RAPIDS can be ignored unless you use cudf/cuml/dask-cuda in this notebook.')

# Quick torch/device check
run([
    sys.executable, '-c',
    'import os, torch; '
    'print(torch.__version__); '
    'print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES")); '
    'print("CUDA available:", torch.cuda.is_available()); '
    'print("GPU count:", torch.cuda.device_count()); '
    '[print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]'
])

## 3) Dataset check (Step 2 skipped by request)

We only verify that the dataset path exists and contains files.

In [ ]:
if USE_SEPARATE:
    for label, p in [('Face', FACE_PATH), ('Fingerprint', FINGERPRINT_PATH)]:
        if not p.exists():
            raise FileNotFoundError(f'{label} dataset path not found: {p}')
        print(f'{label} dataset path exists: {p}')
        sample_files = list(itertools.islice(p.rglob('*'), 15))
        print(f'  Showing up to 15 entries ({len(sample_files)} found):')
        for f in sample_files:
            print('   -', f)
else:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f'Dataset path not found: {DATASET_PATH}')

    print('Dataset path exists:', DATASET_PATH)
    sample_files = list(itertools.islice(DATASET_PATH.rglob('*'), 30))
    print(f'Showing up to 30 entries ({len(sample_files)} found):')
    for p in sample_files:
        print(' -', p)

## 4) Run training (Step 3)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

resume_from = RESUME_FROM
if resume_from is None and AUTO_RESUME:
    import re

    def checkpoint_rank(path):
        match = re.fullmatch(r'checkpoint_phase([12])_epoch(\d+)\.pt', path.name)
        if match:
            phase = int(match.group(1))
            epoch = int(match.group(2))
            if epoch <= 0:
                return -1, -1, -1
            return phase, epoch, 2
        match = re.fullmatch(r'best_model_phase([12])\.pt', path.name)
        if match:
            return int(match.group(1)), 0, 1
        return -1, -1, -1

    candidates = [
        p for p in OUTPUT_DIR.glob('*.pt')
        if checkpoint_rank(p) != (-1, -1, -1)
    ]
    resume_from = max(candidates, key=checkpoint_rank) if candidates else None

train_cmd = [
    sys.executable, 'main.py',
    '--mode', 'train',
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--weight_decay', str(WEIGHT_DECAY),
    '--image_size', str(IMAGE_SIZE),
    '--embed_dim', str(EMBED_DIM),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]

if TIMEOUT_SECONDS is not None:
    train_cmd += ['--timeout', str(TIMEOUT_SECONDS)]

if resume_from is not None:
    resume_from = Path(resume_from)
    if resume_from.exists():
        print('Resuming training from:', resume_from)
        train_cmd += ['--resume', str(resume_from)]
    else:
        raise FileNotFoundError(f'RESUME_FROM checkpoint not found: {resume_from}')
else:
    print('No checkpoint selected; starting training from scratch.')

if USE_SEPARATE:
    train_cmd += [
        '--face_path', str(FACE_PATH),
        '--fingerprint_path', str(FINGERPRINT_PATH),
    ]

import time
import subprocess as _sp

if TIMEOUT_SECONDS is not None:
    print(f'Timeout monitoring: {TIMEOUT_SECONDS}s (≈{TIMEOUT_SECONDS/3600:.1f}h)')
    GRACE_PERIOD = 300  # 5min for the subprocess to save & exit
    start_time = time.time()

    proc = _sp.Popen(train_cmd, cwd=str(SRC_DIR), text=True)

    try:
        while proc.poll() is None:
            elapsed = time.time() - start_time
            if elapsed >= TIMEOUT_SECONDS:
                print(f'\n*** TIMEOUT ({TIMEOUT_SECONDS}s) — sending SIGTERM ***')
                proc.terminate()
                try:
                    proc.wait(timeout=GRACE_PERIOD)
                except _sp.TimeoutExpired:
                    print('*** Grace period expired — sending SIGKILL ***')
                    proc.kill()
                    proc.wait()
                break
            time.sleep(10)
    except KeyboardInterrupt:
        print('\n*** Notebook interrupted — forwarding SIGTERM ***')
        proc.terminate()
        try:
            proc.wait(timeout=GRACE_PERIOD)
        except _sp.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise

    rc = proc.returncode
    if rc != 0:
        print(f'Training process exited with code {rc}')
    else:
        print('Training process exited successfully.')
else:
    rc = run(train_cmd, cwd=str(SRC_DIR), check=False)
    if rc != 0:
        print(f'Training process exited with code {rc} (continuing with saved artifacts).')


## 5) Run evaluation (Step 4)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Resolve best checkpoint: prefer phase2, fall back to phase1
_ckpt_candidates = [
    OUTPUT_DIR / 'best_model_phase2.pt',
    OUTPUT_DIR / 'best_model_phase1.pt',
    OUTPUT_DIR / 'shutdown_checkpoint_phase2.pt',
    OUTPUT_DIR / 'shutdown_checkpoint_phase1.pt',
]
best_checkpoint = next((p for p in _ckpt_candidates if p.exists()), None)
if best_checkpoint is None:
    # last resort: any .pt in OUTPUT_DIR
    _pts = sorted(OUTPUT_DIR.glob('*.pt'))
    best_checkpoint = _pts[-1] if _pts else None
if best_checkpoint is None:
    raise FileNotFoundError(f'No checkpoint found in {OUTPUT_DIR}. Run training first.')
print(f'Using checkpoint: {best_checkpoint}')

if USE_SEPARATE:
    # --- Automated unimodal evaluation (separate-dataset mode) ---
    # Evaluates face encoder on CASIA-WebFace and fingerprint encoder on SOCOFing
    # independently, producing figures/ and tables/ for LaTeX.
    print('Separate dataset mode: running unimodal evaluation via evaluate_separate.py')
    eval_sep_cmd = [
        sys.executable, str(SRC_DIR / 'evaluate_separate.py'),
        '--face_path', str(FACE_PATH),
        '--fingerprint_path', str(FINGERPRINT_PATH),
        '--model_path', str(best_checkpoint),
        '--output_dir', str(RESULTS_DIR),
        '--embed_dim', str(EMBED_DIM),
        '--image_size', str(IMAGE_SIZE),
        '--batch_size', str(BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
        '--device', DEVICE,
    ]
    run(eval_sep_cmd, cwd=str(SRC_DIR))
else:
    MODEL_PATH = OUTPUT_DIR / 'checkpoints' / 'best_model.pth'
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'Expected best model not found: {MODEL_PATH}')

    # Use evaluate.py directly — it produces figures/ and tables/ with PDFs + .tex
    eval_cmd = [
        sys.executable, 'evaluate.py',
        '--model_path', str(MODEL_PATH),
        '--dataset_path', str(DATASET_PATH),
        '--output_dir', str(RESULTS_DIR),
        '--batch_size', str(BATCH_SIZE),
        '--image_size', str(IMAGE_SIZE),
        '--num_workers', str(NUM_WORKERS),
        '--device', DEVICE,
    ]
    run(eval_cmd, cwd=str(SRC_DIR))

## 6) Run visualization (Step 5)

In [ ]:
if USE_SEPARATE:
    print('Separate dataset mode: skipping visualization (no shared test set).')
else:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'Expected best model not found: {MODEL_PATH}')

    viz_cmd = [
        sys.executable, 'main.py',
        '--mode', 'visualize',
        '--model_path', str(MODEL_PATH),
        '--dataset_path', str(DATASET_PATH),
        '--output_dir', str(RESULTS_DIR),
        '--batch_size', str(BATCH_SIZE),
        '--image_size', str(IMAGE_SIZE),
        '--embed_dim', str(EMBED_DIM),
        '--num_workers', str(NUM_WORKERS),
        '--device', DEVICE,
    ]
    run(viz_cmd, cwd=str(SRC_DIR))

## 7) Copy generated files into LaTeX directory (Step 6)

In [ ]:
latex_figures = LATEX_DIR / 'figures'
latex_tables = LATEX_DIR / 'tables'
latex_figures.mkdir(parents=True, exist_ok=True)
latex_tables.mkdir(parents=True, exist_ok=True)

results_figures = RESULTS_DIR / 'figures'
results_tables = RESULTS_DIR / 'tables'

if results_figures.exists():
    for pdf in results_figures.glob('*.pdf'):
        shutil.copy2(pdf, latex_figures / pdf.name)
    print('Copied figures:', [p.name for p in sorted(latex_figures.glob('*'))])
else:
    print('No figures/ directory found in results.')

if results_tables.exists():
    for tex in results_tables.glob('*.tex'):
        shutil.copy2(tex, latex_tables / tex.name)
    print('Copied tables:', [p.name for p in sorted(latex_tables.glob('*'))])
else:
    print('No tables/ directory found in results.')

if USE_SEPARATE:
    print('\nNote: Separate dataset mode — figures/tables are placeholders.')
    print('Replace with real values from unimodal evaluation on each dataset.')

## 8) Placeholder tracking helper (Step 7 support)

This cell counts unreplaced `\textcolor{red}{...}` placeholders in `article.tex` and `memoire.tex`.

In [ ]:
def count_placeholders(tex_path: Path):
    text = tex_path.read_text(encoding='utf-8', errors='ignore')
    needle = 'textcolor{red}'
    return text.count(needle)

for name in ['article.tex', 'memoire.tex']:
    p = LATEX_DIR / name
    if p.exists():
        print(name, '-> placeholders:', count_placeholders(p))
    else:
        print(name, '-> not found')


In [ ]:
# Install pdflatex and dependencies on Kaggle (requires internet enabled in notebook settings)
!apt-get update && apt-get install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended cm-super texlive-bibtex-extra texlive-publishers texlive-lang-french texlive-science


In [ ]:
# Clean up LaTeX build artifacts (aux/bbl/log/out) but keep .tex, .bib, and figure PDFs
import shutil
keep_exts = {'.tex', '.bib', '.pdf', '.cls', '.bst', '.sty'}
keep_dirs = {'figures', 'tables'}
if LATEX_DIR.exists():
    for path in LATEX_DIR.iterdir():
        if path.is_dir():
            if path.name not in keep_dirs:
                shutil.rmtree(path)
        else:
            keep = path.suffix.lower() in keep_exts
            if not keep:
                path.unlink()
    print('Cleaned LaTeX directory (kept .tex, .bib, .pdf, figures/, tables/).')

## 9) Compile LaTeX PDFs (Step 8, optional on Kaggle)

If `pdflatex`/`bibtex` are not available in the Kaggle runtime, this step will fail and can be skipped.

In [ ]:
# Apply local LaTeX fixes to the (older) copies in LATEX_DIR. Idempotent.
#
# Mirrors local commits:
#   0397861 (memoire.tex): algpseudocode->algorithmic, \TO/\margin macros,
#                          duplicate sec:attention label, table column count
#   f4ab398 (article.tex): robustness table column count + multicolumn header
#   e0ae72d (article.tex): \usepackage{threeparttable}

def patch_file(path, transforms, name):
    text = path.read_text(encoding='utf-8', errors='ignore')
    original = text
    applied = 0
    for needle, replacement in transforms:
        if replacement in text:
            continue  # already patched
        if needle not in text:
            print(f'  [skip] {name}: not found: {needle[:50]!r}')
            continue
        text = text.replace(needle, replacement, 1)
        applied += 1
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Patched {name}: {applied} transformation(s) applied')
    else:
        print(f'{name}: no changes needed (already up-to-date)')

memoire_path = LATEX_DIR / 'memoire.tex'
if memoire_path.exists():
    sect_title = "\\section{Module de Fusion par Attention Crois\\'{e}e Modul\\'{e}e par la Qualit\\'{e}}"
    memoire_patches = [
        ("\\usepackage{algpseudocode}",
         "\\usepackage{algorithmic}"),
        ("\\usepackage{etoolbox}\n\n% Fix abstract package conflict with French babel",
         "\\usepackage{etoolbox}\n\n% Custom commands\n\\newcommand{\\TO}{\\textbf{to}}\n\\newcommand{\\margin}{\\ensuremath{m}}\n\n% Fix abstract package conflict with French babel"),
        (sect_title + "\n\\label{sec:attention}",
         sect_title + "\n\\label{sec:fusion-attention}"),
        ("(\\S\\ref{sec:attention})",
         "(\\S\\ref{sec:fusion-attention})"),
        ("\\label{tab:resultats-principaux}\n    \\begin{tabular}{lcccccc}",
         "\\label{tab:resultats-principaux}\n    \\begin{tabular}{lccccc}"),
    ]
    patch_file(memoire_path, memoire_patches, 'memoire.tex')

article_path = LATEX_DIR / 'article.tex'
if article_path.exists():
    article_patches = [
        ("\\usepackage{pifont}\n\n\\begin{document}",
         "\\usepackage{pifont}\n\\usepackage{threeparttable}\n\n\\begin{document}"),
        ("\\begin{tabular}{l | c | c c c c c | c c c}",
         "\\begin{tabular}{l | c | c c c c c c | c c c}"),
        ("& & & & & & & \\multicolumn{3}{c}{\\textbf{Single Modality}} \\\\",
         "& & & & & & & & \\multicolumn{3}{c}{\\textbf{Single Modality}} \\\\"),
    ]
    patch_file(article_path, article_patches, 'article.tex')

print()
print('Files present in LATEX_DIR:')
for p in sorted(LATEX_DIR.glob('*.tex')):
    print(f'  - {p.name} ({p.stat().st_size} bytes)')


In [ ]:
has_pdflatex = shutil.which('pdflatex') is not None
has_bibtex = shutil.which('bibtex') is not None
print('pdflatex available:', has_pdflatex)
print('bibtex available:', has_bibtex)

import subprocess as _sp, time as _time
PASS_TIMEOUT = 300  # seconds per pdflatex/bibtex pass; >0 prevents infinite hangs

def _run_tex(cmd, cwd, log_name):
    """Run one tex pass with timeout + streaming. Capture to log file."""
    log_path = LATEX_DIR / log_name
    print(f'  [{" ".join(cmd)}]  ', end='', flush=True)
    t0 = _time.time()
    try:
        with open(log_path, 'w') as logf:
            proc = _sp.Popen(cmd, cwd=cwd, stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True, encoding='latin-1', errors='replace')
            try:
                for line in proc.stdout:
                    print(line, end='')
                    logf.write(line)
                rc = proc.wait(timeout=PASS_TIMEOUT)
            except _sp.TimeoutExpired:
                proc.kill()
                proc.wait()
                print(f'TIMEOUT after {PASS_TIMEOUT}s', flush=True)
                logf.write(f'\n*** TIMEOUT after {PASS_TIMEOUT}s ***\n')
                return -1
        elapsed = _time.time() - t0
        print(f'done in {elapsed:.1f}s (rc={rc})')
        return rc
    except FileNotFoundError:
        print('binary not found — skipping')
        return -1

def compile_tex(doc_name: str):
    if not has_pdflatex:
        print(f'Skipping {doc_name}: pdflatex not available')
        return
    print(f'\n=== Compiling {doc_name}.tex ===')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass1.log')
    if has_bibtex:
        _run_tex(['bibtex', doc_name], str(LATEX_DIR), f'{doc_name}.bibtex.log')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass2.log')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass3.log')
    pdf = LATEX_DIR / f'{doc_name}.pdf'
    if pdf.exists():
        print(f'  OK: {pdf.name} ({pdf.stat().st_size} bytes)')
    else:
        print(f'  !! {pdf.name} NOT produced — see {LATEX_DIR}/{doc_name}.*.log')

compile_tex('article')
compile_tex('memoire')


## 10) Final artifact summary

In [ ]:
def show_files(title, root: Path, patterns):
    print('\n' + title)
    if not root.exists():
        print('  (missing)')
        return
    found = []
    for pat in patterns:
        found.extend(root.glob(pat))
    found = sorted(set(found))
    if not found:
        print('  (none found)')
    else:
        for p in found:
            print(' -', p)

show_files('Model checkpoints', OUTPUT_DIR, ['**/*.pth'])
show_files('Evaluation/visualization outputs', RESULTS_DIR, ['**/*.json', '**/*.pdf', '**/*.tex'])
show_files('LaTeX artifacts', LATEX_DIR, ['*.tex', '*.pdf', 'figures/*.pdf', 'tables/*.tex'])


In [ ]:
import zipfile
from pathlib import Path
import os

# Zip each top-level folder under /kaggle/working into its own .zip
working_dir = Path('/kaggle/working')

# Skip 'output' (model checkpoints are large and not needed downstream)
EXCLUDE = {'output', '.ipynb_checkpoints'}

for entry in sorted(working_dir.iterdir()):
    if not entry.is_dir() or entry.name.startswith('.') or entry.name in EXCLUDE:
        continue

    archive_path = working_dir / f'{entry.name}.zip'
    print(f'\nZipping {entry.name}/ -> {archive_path.name} ...')

    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(entry):
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            for file in files:
                file_path = Path(root) / file
                arcname = file_path.relative_to(entry.parent)
                zf.write(file_path, arcname)

    size_mb = archive_path.stat().st_size / 1024 / 1024
    print(f'  Created: {archive_path.name} ({size_mb:.1f} MB)')

print('\nAll zips:')
for z in sorted(working_dir.glob('*.zip')):
    mb = z.stat().st_size / 1024 / 1024
    print(f'  {z.name}  ({mb:.1f} MB)')
